# A Simple RAG Demo Project

Before running the notebook, follow the setup instructions in [README.md](README.md).

In [ ]:
# Run once when setting up the project in Colab
#!git clone https://github.com/raivisskadins/simple-rag-demo.git
#%cd simple-rag-demo
#!git pull

# Run once if packages are missing (required also for Colab)
#!pip install -r requirements.txt

# Run to pull latest changes from GitHub
#!git pull

In [ ]:
# ==============================
# 1. Imports
# ==============================

import os
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv, find_dotenv

In [ ]:
# ==============================
# 2. Load environment variables
# ==============================

load_dotenv(find_dotenv(), override=True)

print("GROQ_API_KEY loaded:", os.getenv("GROQ_API_KEY"))

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

## Knowledge Base Loading & Chunking

The knowledge base is three text files covering general facts, Python programming, and Riga, Latvia. Raw text must be split into **chunks** before embedding — the granularity directly affects retrieval quality.

We use a **sentence-level sliding window** (window=3, step=2):
1. Each paragraph is split into sentences on `". "`
2. Consecutive triples of sentences form one chunk
3. The window advances by 2, leaving **1 sentence of overlap** between adjacent chunks

This is better than naive paragraph splitting because sentences near boundaries appear in at least one chunk (no information lost at edges), and fixed-size windows produce more uniform embeddings than variable-length paragraphs.

In [ ]:
# ==============================
# 3. Load knowledge base
# ==============================

def make_chunks(text, window=3, step=2):
    """Sentence-level sliding window chunker."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    sentences = []
    for para in paragraphs:
        sents = [s.strip() for s in para.split(". ") if s.strip()]
        sentences.extend(sents)
    result = []
    for i in range(0, len(sentences), step):
        chunk_sents = sentences[i : i + window]
        if chunk_sents:
            result.append(". ".join(chunk_sents))
    return result

kb_files = [
    "./data/knowledge.txt",
    "./data/python_faq.txt",
    "./data/riga_facts.txt",
]

chunks = []
for filepath in kb_files:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    chunks.extend(make_chunks(text))

print(f"Total chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"[{i}]", c)

## Creating Embeddings & Building the FAISS Index

Each chunk is converted into a **dense vector embedding** using `paraphrase-multilingual-mpnet-base-v2` — a sentence-transformer model that maps semantically similar text to nearby points in a high-dimensional vector space. Embeddings are L2-normalised so cosine similarity equals inner product.

The vectors are loaded into a **FAISS `IndexFlatIP`** (exact inner-product search). FAISS is optimised for fast nearest-neighbour lookup at scale, making it straightforward to swap in approximate indices later if the knowledge base grows large.

In [ ]:
# ==============================
# 4. Create embeddings
# ==============================

def get_embedding(text):
    emb = embedding_model.encode(text)
    emb = emb / np.linalg.norm(emb)
    return emb

chunk_embeddings = np.array(
    [get_embedding(chunk) for chunk in chunks]
).astype("float32")
embedding_dim = chunk_embeddings.shape[1]



# ==============================
# 5. Create FAISS index
# ==============================

index = faiss.IndexFlatIP(embedding_dim)
# remove previously added content
index.reset()
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)

## Retrieval: Embedding the Query & Searching the Index

To answer a question, we embed it with the **same model** used for the knowledge base — this ensures the question and the chunks live in the same vector space, so distances are meaningful. A FAISS inner-product search returns the **k=3 chunks** most semantically similar to the question.

Semantic search works even when the question uses different words than the source text, unlike keyword search (BM25/TF-IDF), which requires exact lexical overlap. The top-3 chunks are concatenated and passed to the LLM as context.

In [ ]:
# ==============================
# 6. User question
# ==============================

#question = "Which ocean is the largest on Earth?"
question = "Is Great Wall of China visible from the Moon?"

print("Question:", question)


# ==============================
# 7. Embed the question
# ==============================

question_embedding = np.array(
    [get_embedding(question)]
).astype("float32")

# You can print out the embedding just in case you want to see how it looks
#print("Question Embedding:", question_embedding)


## Improvement 1: Top-k Retrieval (k=3)

Increasing the number of retrieved chunks from **k=2 to k=3** gives the LLM broader context for each query. With only two chunks, relevant information that is spread across multiple paragraphs can be missed entirely. Retrieving three chunks improves recall — especially for questions that touch on multiple aspects of the knowledge base — at negligible cost since FAISS search is fast.

In [ ]:
# ==============================
# 8. Retrieve top 3 chunks
# ==============================

k = 3

distances, indices = index.search(question_embedding, k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved context:")
for c in retrieved_chunks:
    print("-", c)

## Prompt Construction

Before calling the LLM we assemble a structured prompt from the retrieved chunks and the user question.

**Improvement: system + user message split.** The original design crammed instructions, context, and the question into a single user message. Using a dedicated **system message** for behavioural instructions and a **user message** for content is the correct pattern for chat models:
- The system message tells the model *how* to behave — stay grounded, no outside knowledge, explicit fallback phrasing
- The user message delivers only what the model needs to reason over

This separation reduces hallucination risk and makes the fallback (`"There is no information available."`) more reliably honoured.

In [ ]:
# ==============================
# 9. Create RAG prompt
# ==============================

context = "\n\n".join(retrieved_chunks)

system_message = (
    "You are a precise question-answering assistant. "
    "Answer the user's question using ONLY the information provided in the context. "
    "Do not use any outside knowledge or make assumptions beyond what is explicitly stated. "
    "Keep your answer concise — one short paragraph at most. "
    "If the context does not contain enough information to answer, respond with exactly: "
    "'There is no information available.'"
)

user_message = f"""Context:
{context}

Question: {question}"""

#print("System:", system_message)
#print("User:", user_message)

## LLM Generation

With the context retrieved and the prompt assembled, we call the **Groq API** using `llama-3.3-70b-versatile`. Groq's LPU inference delivers very low latency, making the end-to-end RAG loop feel near-instant. `temperature=0` ensures deterministic, reproducible answers — important for a grounded Q&A system where we want the model to report facts, not generate variations.

In [ ]:
# ==============================
# 10. Call Groq LLM to generate the answer
# ==============================

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    temperature=0
)

print("\nLLM Answer:")
print(response.choices[0].message.content)

## Test Questions

Three pre-defined questions exercise different parts of the knowledge base — Python virtual environments, Riga's geography, and pip package management. Running these after any change to chunking, retrieval, or prompting gives a quick regression check that the pipeline still produces correct, grounded answers.

In [ ]:
# ==============================
# 11. Test questions
# ==============================

test_system_message = (
    "You are a precise question-answering assistant. "
    "Answer the user's question using ONLY the information provided in the context. "
    "Do not use any outside knowledge or make assumptions beyond what is explicitly stated. "
    "Keep your answer concise — one short paragraph at most. "
    "If the context does not contain enough information to answer, respond with exactly: "
    "'There is no information available.'"
)

test_questions = [
    "What is a Python virtual environment and how do you create one?",
    "What is the population of Riga and what river flows through it?",
    "How do you install a Python package using pip?",
]

for test_q in test_questions:
    q_emb = np.array([get_embedding(test_q)]).astype("float32")
    _, idxs = index.search(q_emb, 3)
    ctx = "\n\n".join([chunks[i] for i in idxs[0]])

    test_user_message = f"""Context:
{ctx}

Question: {test_q}"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": test_system_message},
            {"role": "user", "content": test_user_message}
        ],
        temperature=0
    )

    print(f"Q: {test_q}")
    print(f"A: {resp.choices[0].message.content}")
    print()

## Interactive Q&A Loop

A live terminal loop that lets you ask arbitrary questions against the knowledge base in real time. The full RAG pipeline runs on each input — embed the query, FAISS retrieval (k=3), prompt construction with system + user roles, and Groq generation. Type `quit` or `exit` to stop.

In [ ]:
# ==============================
# 12. Interactive Q&A loop
# ==============================

rag_system_message = (
    "You are a precise question-answering assistant. "
    "Answer the user's question using ONLY the information provided in the context. "
    "Do not use any outside knowledge or make assumptions beyond what is explicitly stated. "
    "Keep your answer concise — one short paragraph at most. "
    "If the context does not contain enough information to answer, respond with exactly: "
    "'There is no information available.'"
)

print("RAG Interactive Q&A — type 'quit' or 'exit' to stop.\n")

while True:
    user_q = input("Your question: ").strip()
    if user_q.lower() in ("quit", "exit"):
        print("Goodbye!")
        break
    if not user_q:
        continue

    q_emb = np.array([get_embedding(user_q)]).astype("float32")
    _, idxs = index.search(q_emb, 3)
    ctx = "\n\n".join([chunks[i] for i in idxs[0]])

    rag_user_message = f"""Context:
{ctx}

Question: {user_q}"""

    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": rag_system_message},
            {"role": "user", "content": rag_user_message},
        ],
        temperature=0,
    )

    print(f"Answer: {resp.choices[0].message.content}\n")